**RANDOM FOREST AND GRADIENT BOOSTING**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("train.csv")

# Same fills as Day 17
no_feature_cols = ["PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
                   "GarageType", "GarageFinish", "GarageQual", "GarageCond",
                   "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
                   "MasVnrType"]
df[no_feature_cols] = df[no_feature_cols].fillna("None")
df["MasVnrArea"] = df["MasVnrArea"].fillna(0)
df["GarageYrBlt"] = df["GarageYrBlt"].fillna(0)
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# Same split as every earlier day
train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.copy()
test = test.copy()
median_frontage = train["LotFrontage"].median()
train["LotFrontage"] = train["LotFrontage"].fillna(median_frontage)
test["LotFrontage"] = test["LotFrontage"].fillna(median_frontage)

y_train = train["SalePrice"]
y_test = test["SalePrice"]

X_train = pd.get_dummies(train.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test = pd.get_dummies(test.drop(columns=["SalePrice", "Id"]), dtype=int)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

def score(model, name):
  model.fit(X_train, y_train)
  p = model.predict(X_test)
  print(name, "| MAE:", round(mean_absolute_error(y_test, p)), "| R2:", round(r2_score(y_test, p), 2))
  return model

keep = [c for c in X_train.columns if c != "PoolArea" and not c.startswith("PoolQC")]
lin = LinearRegression().fit(X_train[keep], y_train)
p_lin = lin.predict(X_test[keep])
print("Linear, all columns without pool | MAE:", round(mean_absolute_error(y_test, p_lin)), "| R2:", round(r2_score(y_test, p_lin), 2))

rf = score(RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1), "Random forest")
gb = score(GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42), "Gradient boosting")




Linear, all columns without pool | MAE: 19245 | R2: 0.88
Random forest | MAE: 17426 | R2: 0.89
Gradient boosting | MAE: 16626 | R2: 0.9


**WHAT DOES THE MODEL USE**

In [8]:
imp = pd.Series(gb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print((imp.head(10) * 100).round(1))
print()
print("Share of importance held by all pool columns:",
      round(imp[[c for c in imp.index if c.startswith("Pool")]].sum() * 100, 2), "%")

OverallQual    50.3
GrLivArea      15.1
GarageCars      4.7
BsmtFinSF1      3.6
TotalBsmtSF     3.5
1stFlrSF        3.1
2ndFlrSF        2.6
LotArea         1.8
YearBuilt       1.7
BsmtQual_Ex     1.4
dtype: float64

Share of importance held by all pool columns: 0.19 %


*The five most important columns for gradient boosting are OverallQual (50.3%), GrLivArea (15.1%), GarageCars (4.7%), BsmtFinSF1 (3.6%), and TotalBsmtSF (3.5%). The pool columns together hold 0.19%, and the lowest prediction is 57,761, so the trees avoid the negative prices that broke the linear model.*

In [9]:
p_gb = gb.predict(X_test)
print("Lowest predicted price:", round(p_gb.min()))

check = pd.DataFrame({"actual": y_test.values, "gb": p_gb.round(), "linear": p_lin.round()},
                     index=y_test.index)
check["Neighborhood"] = test["Neighborhood"]
check.sort_values("actual", ascending=False).head(5)

Lowest predicted price: 57761


,actual,gb,linear,Neighborhood
691,755000,645308.0,585274.0,NoRidge
898,611657,399104.0,471095.0,NridgHt
1046,556581,444963.0,425105.0,StoneBr
1243,465000,404418.0,414481.0,NridgHt
591,451950,424992.0,415875.0,NridgHt


*On the five most expensive test houses, both gradient boosting and the linear model underpredict. Boosting is closer on three (including the 755,000 NoRidge house), but one NridgHt house (actual 611,657) is predicted at 399,104. Luxury homes remain the hardest to price.*